# Train DistilBERT

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'load_data.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the 6Emotions project or a model folder.')
sys.path.insert(0, str(PROJECT_ROOT))
from src.load_data import load_model_data, TARGET_LABELS
from src.evaluate import save_confusion_matrices

RANDOM_STATE = 42
JOY_CAP = 12000
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
RUN_DIR = PROJECT_ROOT / 'models' / 'distilbert' / 'runs'
MODEL_DIR = PROJECT_ROOT / 'models' / 'distilbert' / 'artifacts'
RESULTS_DIR = PROJECT_ROOT / 'models' / 'distilbert' / 'results'

LABEL_TO_ID = {label: idx for idx, label in enumerate(TARGET_LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

/Users/hayden/HaydensOffice/Projects/6Emotions/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = load_model_data(base_path=PROJECT_ROOT / 'data', random_state=RANDOM_STATE, joy_cap=JOY_CAP)
full_train_df = data['full_train']
full_test_df = data['full_test']

# Make sure there are no missing values and that the labels are correct
assert not full_train_df[['text', 'label']].isna().any().any()
assert not full_test_df[['text', 'label']].isna().any().any()
assert set(full_train_df['label']) == set(TARGET_LABELS)
assert set(full_test_df['label']) == set(TARGET_LABELS)
print(f'Training size: {len(full_train_df)}')
print(f'Test size:     {len(full_test_df)}')
print(full_train_df['label'].value_counts().reindex(TARGET_LABELS))

Training size: 39718
Test size:     5421
label
sadness      7966
joy         12000
love         3904
anger        7756
fear         2784
surprise     5308
Name: count, dtype: int64


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_frame(frame):
    dataset = Dataset.from_dict({
        'text': frame['text'].astype(str).tolist(),
        'labels': frame['label'].map(LABEL_TO_ID).astype(int).tolist(),
    })
    return dataset.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=['text'],
    )

train_dataset = tokenize_frame(full_train_df)
test_dataset = tokenize_frame(full_test_df)
print(f'Tokenized {len(train_dataset)} training and {len(test_dataset)} test rows.')

Map: 100%|██████████| 5421/5421 [00:00<00:00, 116258.24 examples/s]

Tokenized 39718 training and 5421 test rows.


In [4]:
def balanced_weights(labels):
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=len(TARGET_LABELS))
    if np.any(counts == 0):
        raise ValueError('The training data must contain all six labels.')
    return torch.tensor(len(labels) / (len(TARGET_LABELS) * counts), dtype=torch.float32)


def weighted_loss_for(labels):
    weights = balanced_weights(labels)
    def compute_loss(outputs, batch_labels, num_items_in_batch=None):
        return F.cross_entropy(
            outputs.logits,
            batch_labels,
            weight=weights.to(outputs.logits.device),
        )
    return compute_loss


def new_model():
    set_seed(RANDOM_STATE)
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(TARGET_LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
    )


def new_trainer(model, training_rows):
    args = TrainingArguments(
        output_dir=str(RUN_DIR),
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        optim='adamw_torch',
        save_strategy='no',
        logging_strategy='epoch',
        report_to='none',
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        use_cpu=(DEVICE == 'cpu'),
        dataloader_pin_memory=False,
    )
    return Trainer(
        model=model,
        args=args,
        train_dataset=training_rows,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_loss_func=weighted_loss_for(training_rows['labels']),
    )

In [ ]:
final_trainer = new_trainer(new_model(), train_dataset)
final_trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8027.68it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


## Held-out test evaluation

Run this after training. The test data is not used during training.

In [ ]:
test_output = final_trainer.predict(test_dataset)
y_true = np.asarray(test_output.label_ids)
y_pred = np.argmax(test_output.predictions, axis=1)
test_accuracy = float(accuracy_score(y_true, y_pred))
print(f'Final test accuracy: {test_accuracy:.4f}')
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(TARGET_LABELS))),
    target_names=TARGET_LABELS,
))

In [ ]:
cm = save_confusion_matrices(
    [ID_TO_LABEL[i] for i in y_true],
    [ID_TO_LABEL[i] for i in y_pred],
    TARGET_LABELS, RESULTS_DIR, 'DistilBERT',
)

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

summary = {
    'base_model': MODEL_NAME,
    'random_state': RANDOM_STATE,
    'training_rows': len(full_train_df),
    'test_rows': len(full_test_df),
    'joy_cap': JOY_CAP,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'max_length': MAX_LENGTH,
    'learning_rate': LEARNING_RATE,
    'test_accuracy': test_accuracy,
    'labels': TARGET_LABELS,
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(f'Model and tokenizer saved to {MODEL_DIR}; training summary saved to {RESULTS_DIR}')